# E011 — Image PCA n_components sweep

E007 chose PCA 50 components without ablation. This experiment sweeps
`n_pca ∈ {20, 30, 50, 75, 100, 150}` with the same +All augmentation
(flip + brightness [0.7,1.3] + noise σ=15) and LOSO CV (seed=67).

**Methodological guard:**
- Scaler + PCA fitted ONLY on augmented train fold, applied to val
- Val fold always uses original images (no augmentation)
- One axis changed vs E007: only n_pca varies

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path("../src").resolve()))

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve
from scipy.stats import norm as scipy_norm
import pandas as pd

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

COLORS = {
    "target":    "#E74C3C",
    "nontarget": "#2E86AB",
    "green":     "#27AE60",
    "winner":    "#F39C12",
}
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

DATA = Path("../data").resolve()
manifest = load_manifest(DATA)
y_all = manifest["label"].to_numpy()
SEED = 67
print(f"{len(manifest)} samples — {manifest.label.sum()} target, {(manifest.label==0).sum()} non-target")

## 1. Helper functions

Same helpers as E007: image loading, augmentation functions.

In [ ]:
def find_png(stem: str, data_dir: Path) -> Path:
    for sf in ["target_train", "target_dev", "non_target_train", "non_target_dev"]:
        p = data_dir / sf / (stem + ".png")
        if p.exists():
            return p
    raise FileNotFoundError(stem)

def load_image(png_path: Path) -> np.ndarray:
    """Load PNG → RGB mean → flatten. Returns (6400,) in [0, 255]."""
    img = np.array(Image.open(png_path).convert("RGB"), dtype=np.float32)
    return img.mean(axis=2).flatten()

def load_images_original(df: pd.DataFrame, data_dir: Path) -> np.ndarray:
    return np.stack([load_image(find_png(row["stem"], data_dir)) for _, row in df.iterrows()])


# --- Augmentation functions ---

def aug_flip(x: np.ndarray) -> np.ndarray:
    """Horizontal flip of 80×80 image."""
    return x.reshape(80, 80)[:, ::-1].flatten()

def aug_brightness(x: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Random brightness scale in [0.7, 1.3]."""
    factor = rng.uniform(0.7, 1.3)
    return np.clip(x * factor, 0, 255)

def aug_noise_img(x: np.ndarray, rng: np.random.Generator, sigma: float = 15.0) -> np.ndarray:
    """Additive Gaussian noise σ=15."""
    return np.clip(x + rng.normal(0, sigma, x.shape), 0, 255)

def load_images_augmented(df: pd.DataFrame, data_dir: Path, seed: int) -> tuple:
    """
    Load images and apply +All augmentation (flip + brightness + noise).
    Returns (X_aug, y_aug) = original + 3 augmented copies per sample.
    NEVER call this on val data.
    """
    rng = np.random.default_rng(seed)
    X_orig = load_images_original(df, data_dir)
    y_orig = df["label"].to_numpy()

    aug_X, aug_y = [], []
    for xi, yi in zip(X_orig, y_orig):
        aug_X.append(aug_flip(xi))
        aug_y.append(yi)
        aug_X.append(aug_brightness(xi, rng))
        aug_y.append(yi)
        aug_X.append(aug_noise_img(xi, rng))
        aug_y.append(yi)

    X_aug = np.vstack([X_orig, np.stack(aug_X)])
    y_aug = np.concatenate([y_orig, np.array(aug_y)])
    return X_aug, y_aug

print("Helper functions defined.")

## 2. PCA sweep — LOSO cross-validation

For each `n_pca` value: run 3-fold LOSO CV with +All augmentation on train fold.
Report per-fold EER and mean±std.

In [ ]:
N_PCA_VALUES = [20, 30, 50, 75, 100, 150]
C_LOGREG = 1.0

sweep_results = {}   # n_pca → list of fold dicts
sweep_oof     = {}   # n_pca → oof_scores array

for n_pca in N_PCA_VALUES:
    print(f"\n{'='*50}")
    print(f"n_pca = {n_pca}")
    print('='*50)

    oof_scores   = np.full(len(manifest), np.nan)
    fold_results = []

    for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
        train_df = manifest.loc[train_idx]
        val_df   = manifest.loc[val_idx]
        y_val    = val_df["label"].to_numpy()

        # Augment train fold only
        X_train, y_train_aug = load_images_augmented(train_df, DATA, seed=SEED + fold_id)

        # Val: originals only, always
        X_val = load_images_original(val_df, DATA)

        # Scaler + PCA fitted on augmented train
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_val_s   = scaler.transform(X_val)

        pca = PCA(n_components=n_pca, random_state=SEED)
        X_train_pca = pca.fit_transform(X_train_s)
        X_val_pca   = pca.transform(X_val_s)

        clf = LogisticRegression(C=C_LOGREG, max_iter=1000, random_state=SEED)
        clf.fit(X_train_pca, y_train_aug)

        val_scores = clf.decision_function(X_val_pca)
        oof_scores[val_idx] = val_scores

        eer, _     = compute_eer(val_scores[y_val==1], val_scores[y_val==0])
        min_dcf, _ = compute_min_dcf(val_scores[y_val==1], val_scores[y_val==0])
        fold_results.append({"fold": fold_id, "eer": eer, "min_dcf": min_dcf,
                              "n_train": len(X_train)})
        print(f"  Fold {fold_id}: train={len(X_train)} (aug), val={len(X_val)} → "
              f"EER={eer*100:.2f}%, min-DCF={min_dcf:.4f}")

    sweep_results[n_pca] = fold_results
    sweep_oof[n_pca]     = oof_scores.copy()

print("\nSweep complete.")

## 3. Results table

In [ ]:
print(f"{'n_pca':>6} {'F0 EER':>8} {'F1 EER':>8} {'F2 EER':>8} {'Mean':>8} {'Std':>8} {'min-DCF':>9}")
print("-" * 62)

summary = []
for n_pca in N_PCA_VALUES:
    fold_results = sweep_results[n_pca]
    eers   = [r["eer"]*100 for r in fold_results]
    dcfs   = [r["min_dcf"] for r in fold_results]
    mean_e = np.mean(eers)
    std_e  = np.std(eers)
    mean_d = np.mean(dcfs)
    marker = "  ← E007 baseline" if n_pca == 50 else ""
    print(f"{n_pca:>6} {eers[0]:>8.2f} {eers[1]:>8.2f} {eers[2]:>8.2f} "
          f"{mean_e:>8.2f} {std_e:>8.2f} {mean_d:>9.4f}{marker}")
    summary.append({"n_pca": n_pca, "f0": eers[0], "f1": eers[1], "f2": eers[2],
                    "mean": mean_e, "std": std_e, "min_dcf": mean_d})

print("-" * 62)
best = min(summary, key=lambda x: x["mean"])
print(f"Winner: n_pca={best['n_pca']}  EER={best['mean']:.2f}±{best['std']:.2f}%  min-DCF={best['min_dcf']:.4f}")

# OOF overall for best n_pca
oof_best = sweep_oof[best["n_pca"]]
eer_oof, _   = compute_eer(oof_best[y_all==1], oof_best[y_all==0])
dcf_oof, thr = compute_min_dcf(oof_best[y_all==1], oof_best[y_all==0])
print(f"Best OOF overall: EER={eer_oof*100:.2f}%, min-DCF={dcf_oof:.4f}, threshold={thr:.3f}")

## 4. Plots

Two charts:
1. Mean EER vs n_pca with error bars (±std), winner marked
2. Per-fold bar chart for all n_pca values

In [ ]:
# --- Plot 1: mean EER vs n_pca with error bars ---
pca_vals = [s["n_pca"] for s in summary]
means    = [s["mean"]  for s in summary]
stds     = [s["std"]   for s in summary]
dcfs_mean= [s["min_dcf"] for s in summary]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.errorbar(pca_vals, means, yerr=stds, fmt="o-", color="#2E86AB",
            capsize=6, capthick=2, elinewidth=2, lw=2, markersize=8, label="EER mean ± std")

best_idx = np.argmin(means)
ax.scatter([pca_vals[best_idx]], [means[best_idx]], s=200, color=COLORS["winner"],
           zorder=5, label=f"Winner: n_pca={pca_vals[best_idx]}")
ax.annotate(f"  n_pca={pca_vals[best_idx]}\n  EER={means[best_idx]:.2f}±{stds[best_idx]:.2f}%",
            xy=(pca_vals[best_idx], means[best_idx]),
            xytext=(pca_vals[best_idx]+5, means[best_idx]+0.3),
            fontsize=9, color=COLORS["winner"])

# Mark E007 baseline (n_pca=50)
ax.axvline(50, color="gray", ls="--", lw=1.5, alpha=0.6, label="E007 baseline (n_pca=50)")

ax.set_xlabel("n_pca (PCA components)")
ax.set_ylabel("EER mean ± std [%]")
ax.set_title("EER vs PCA dimensionality (E011)")
ax.legend(fontsize=9)
ax.set_xticks(pca_vals)

# Secondary axis: min-DCF
ax2 = axes[1]
ax2.plot(pca_vals, dcfs_mean, "s--", color="#8E44AD", lw=2, markersize=8, label="mean min-DCF")
ax2.scatter([pca_vals[np.argmin(dcfs_mean)]], [min(dcfs_mean)], s=200,
            color=COLORS["winner"], zorder=5,
            label=f"Winner min-DCF: n_pca={pca_vals[np.argmin(dcfs_mean)]}")
ax2.axvline(50, color="gray", ls="--", lw=1.5, alpha=0.6, label="E007 baseline (n_pca=50)")
ax2.set_xlabel("n_pca (PCA components)")
ax2.set_ylabel("mean min-DCF")
ax2.set_title("min-DCF vs PCA dimensionality (E011)")
ax2.legend(fontsize=9)
ax2.set_xticks(pca_vals)

plt.suptitle("E011 — PCA sweep: EER and min-DCF vs n_components", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 2: per-fold bar chart for all n_pca values ---
n_configs = len(N_PCA_VALUES)
x = np.arange(3)   # 3 folds
width = 0.12

# Color palette for n_pca values
cmap = plt.cm.viridis
pca_colors = [cmap(i / (n_configs - 1)) for i in range(n_configs)]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
for i, (n_pca, color) in enumerate(zip(N_PCA_VALUES, pca_colors)):
    fold_results = sweep_results[n_pca]
    eers = [r["eer"]*100 for r in fold_results]
    offset = (i - n_configs/2 + 0.5) * width
    bars = ax.bar(x + offset, eers, width,
                  label=f"n_pca={n_pca}", color=color, alpha=0.85)
    # Highlight winner n_pca bars
    if n_pca == best["n_pca"]:
        for bar in bars:
            bar.set_edgecolor("gold")
            bar.set_linewidth(2.5)

ax.set_xticks(x)
ax.set_xticklabels(["Fold 0", "Fold 1", "Fold 2"])
ax.set_ylabel("EER [%]")
ax.set_title("Per-fold EER — all n_pca values")
ax.legend(fontsize=8, ncol=2)

# Mean ± std bars
ax = axes[1]
bars = ax.bar(range(n_configs), means, color=pca_colors, alpha=0.85,
              yerr=stds, capsize=6, error_kw=dict(elinewidth=2))
for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, m + s + 0.05,
            f"{m:.2f}\n±{s:.2f}", ha="center", fontsize=8)

ax.set_xticks(range(n_configs))
ax.set_xticklabels([f"n={v}" for v in N_PCA_VALUES], fontsize=9)
ax.set_ylabel("EER mean ± std [%]")
ax.set_title("Mean ± std across folds — all n_pca values")

# Highlight winner
bars[best_idx].set_edgecolor("gold")
bars[best_idx].set_linewidth(3)
ax.annotate("winner", xy=(best_idx, means[best_idx] - stds[best_idx] - 0.15),
            ha="center", fontsize=9, color="goldenrod", fontweight="bold")

plt.suptitle("E011 — Per-fold and mean EER comparison across n_pca", fontsize=13)
plt.tight_layout()
plt.show()

## 5. Summary and recommendation

In [ ]:
print("=" * 60)
print("E011 — PCA sweep summary")
print("=" * 60)
print()
print(f"{'n_pca':>6} | {'Mean EER [%]':>12} | {'Std EER [%]':>11} | {'min-DCF':>8}")
print("-" * 48)
for s in summary:
    winner_mark = " << WINNER" if s["n_pca"] == best["n_pca"] else ""
    e007_mark   = " (E007)" if s["n_pca"] == 50 else ""
    print(f"{s['n_pca']:>6} | {s['mean']:>12.2f} | {s['std']:>11.2f} | {s['min_dcf']:>8.4f}{winner_mark}{e007_mark}")

print()
print(f"Winner: n_pca={best['n_pca']}, EER={best['mean']:.2f}±{best['std']:.2f}%, min-DCF={best['min_dcf']:.4f}")
oof_best = sweep_oof[best["n_pca"]]
eer_oof, _   = compute_eer(oof_best[y_all==1], oof_best[y_all==0])
dcf_oof, thr = compute_min_dcf(oof_best[y_all==1], oof_best[y_all==0])
print(f"Winner OOF: EER={eer_oof*100:.2f}%, min-DCF={dcf_oof:.4f}, threshold={thr:.3f}")

if best["n_pca"] == 50:
    print()
    print("Conclusion: E007 choice (n_pca=50) confirmed as optimal. No change needed.")
else:
    improvement = summary[[s["n_pca"] for s in summary].index(50)]["mean"] - best["mean"]
    print()
    print(f"Conclusion: n_pca={best['n_pca']} outperforms E007 baseline by {improvement:.2f}% EER.")
    print(f"Recommendation: update image flagship to n_pca={best['n_pca']}.")